# M5 Forecasting — Notebook 4: Inventory Optimization & Dollar Impact

**Pipeline:** EDA → Feature Engineering → Forecasting → **Optimization**

| Module | Decision | Method |
|---|---|---|
| **1. Newsvendor** | How much to order? | Critical ratio = c_u / (c_u + c_o) |
| **2. Budget Allocation** | How to split budget across stores? | Greedy marginal revenue per dollar |
| **3. Markdown Scheduling** | Which items to discount? | Price-elasticity clearance model |

**Prerequisite:** Run `03_forecasting.ipynb` first.


In [ ]:
import sys, pathlib

SRC_ROOT = '/kaggle/input/datasets/youssefmousaaid/m5-forecast-optimize-src-files'
DATA_DIR = '/kaggle/input/competitions/m5-forecasting-accuracy/'

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f'src/ present : {(pathlib.Path(SRC_ROOT) / "src").exists()}')
print(f'DATA_DIR     : {DATA_DIR}')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.optimization.newsvendor import (
    CostParams, optimal_order_quantity, expected_cost,
    inventory_dollar_impact, sensitivity_analysis,
)
from src.optimization.budget_alloc import (
    BudgetParams, aggregate_by_store, greedy_budget_allocation, budget_dollar_impact,
)
from src.optimization.markdown import (
    MarkdownParams, demand_lift, optimal_markdown, markdown_dollar_impact,
    MARKDOWN_DEPTHS,
)

COLORS = {'green':'#48bb78','blue':'#63b3ed','orange':'#f6ad55',
          'red':'#fc8181','gray':'#718096','purple':'#b794f4'}
plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a202c',
    'axes.edgecolor':'#2d3748','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#a0aec0','ytick.color':'#a0aec0',
    'text.color':'#e2e8f0','grid.color':'#2d3748',
})

def fmt(v):
    if abs(v) >= 1e9: return f'${v/1e9:.1f}B'
    if abs(v) >= 1e6: return f'${v/1e6:.1f}M'
    if abs(v) >= 1e3: return f'${v/1e3:.0f}K'
    return f'${v:.0f}'

print('Imports OK ✓')


## Load Forecasts

In [ ]:
fc = pd.read_parquet('/kaggle/working/forecasts.parquet')
fc['date'] = pd.to_datetime(fc['date'])
print(f'Rows    : {len(fc):,}')
print(f'Columns : {list(fc.columns)}')
fc[['q10','q50','q90','q_star','inventory']].describe().round(2)


---
## Module 1 — Newsvendor Inventory Optimization

In [ ]:
cost_p = CostParams()
print(f'c_o (holding/unit/day)  : ${cost_p.c_o:.4f}')
print(f'c_u (stockout/unit)     : ${cost_p.c_u:.4f}')
print(f'Critical Ratio          : {cost_p.critical_ratio:.4f}')
print(f'Service Level Target    : {cost_p.critical_ratio*100:.1f}th percentile')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sample = fc.sample(min(2000, len(fc)), random_state=42)
axes[0].scatter(sample['q50'], sample['q_star'], alpha=0.3, s=10,
                c=sample['q90'], cmap='viridis')
axes[0].plot([0, sample['q50'].max()],[0, sample['q50'].max()],
             color=COLORS['gray'], lw=1, linestyle='--', label='Q*=q50')
axes[0].set_title('Optimal Order Q* vs Median Forecast')
axes[0].set_xlabel('q50'); axes[0].set_ylabel('Q*'); axes[0].legend()

excess = fc['q_star'] - fc['q50']
axes[1].hist(excess, bins=60, color=COLORS['purple'], edgecolor='none', alpha=0.85)
axes[1].axvline(0, color=COLORS['red'], lw=1.5, linestyle='--')
axes[1].set_title('Q* - q50  (positive = buffer above median)')
axes[1].set_xlabel('Units above median')
pct_above = (excess > 0).mean() * 100
axes[1].text(0.65, 0.85, f'{pct_above:.0f}% of items\nabove median',
             transform=axes[1].transAxes, color=COLORS['green'], fontsize=10)

plt.suptitle('Newsvendor: Why Q* > q50', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
nv = inventory_dollar_impact(fc)
print('=== Newsvendor Dollar Impact ===')
for k, v in nv.items():
    print(f'  {k:<45}: {fmt(v) if isinstance(v,(int,float)) else v}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(['Naive (q50)','Newsvendor Q*'],
            [nv['naive_total_cost_usd'], nv['optimal_total_cost_usd']],
            color=[COLORS['red'],COLORS['green']], edgecolor='none')
for i, v in enumerate([nv['naive_total_cost_usd'], nv['optimal_total_cost_usd']]):
    axes[0].text(i, v*1.01, fmt(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Inventory Cost: Naive vs Optimal (28-day)')
axes[0].set_ylabel('Cost ($)')

sens = sensitivity_analysis(fc, c_u_range=[0.5, 1.0, 2.0, 3.0, 4.0, cost_p.c_u])
axes[1].plot(sens['c_u'], sens['critical_ratio']*100,
             color=COLORS['blue'], marker='o', lw=2)
axes[1].axvline(cost_p.c_u, color=COLORS['orange'], lw=1.5, linestyle='--',
                label=f'Default c_u={cost_p.c_u:.2f}')
axes[1].set_title('Sensitivity: Service Level vs Stockout Cost')
axes[1].set_xlabel('Stockout cost per unit ($)'); axes[1].set_ylabel('Service Level (%)')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


---
## Module 2 — Store Budget Allocation

In [ ]:
ba_params = BudgetParams(budget_usd=500_000)
store_df  = aggregate_by_store(fc)
print(f'Budget: ${ba_params.budget_usd:,.0f}')
print()
print(store_df[['store_id','q10','q50','q90']].set_index('store_id').round(0))


In [ ]:
alloc = greedy_budget_allocation(store_df, ba_params)
ba    = budget_dollar_impact(fc, params=ba_params)
print('=== Budget Allocation Dollar Impact ===')
for k, v in ba.items():
    if k != 'store_allocation_df':
        print(f'  {k:<45}: {fmt(v) if isinstance(v,(int,float)) else v}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = range(len(alloc)); w = 0.35

axes[0].bar([i-w/2 for i in x], alloc['alloc_naive_units'],   w, color=COLORS['gray'],  label='Naive', edgecolor='none')
axes[0].bar([i+w/2 for i in x], alloc['alloc_optimal_units'], w, color=COLORS['green'], label='Optimal', edgecolor='none')
axes[0].set_xticks(x); axes[0].set_xticklabels(alloc['store_id'], rotation=45, fontsize=8)
axes[0].set_title('Units Allocated per Store'); axes[0].legend(fontsize=8)

axes[1].bar([i-w/2 for i in x], alloc['revenue_naive_usd'],   w, color=COLORS['gray'], edgecolor='none')
axes[1].bar([i+w/2 for i in x], alloc['revenue_optimal_usd'], w, color=COLORS['blue'], edgecolor='none')
axes[1].set_xticks(x); axes[1].set_xticklabels(alloc['store_id'], rotation=45, fontsize=8)
axes[1].set_title('Expected Revenue per Store ($)')

delta = alloc['alloc_optimal_units'] - alloc['alloc_naive_units']
axes[2].bar(alloc['store_id'], delta,
            color=[COLORS['green'] if d > 0 else COLORS['red'] for d in delta], edgecolor='none')
axes[2].axhline(0, color=COLORS['gray'], lw=1)
axes[2].set_title('Reallocation Delta (Optimal - Naive)')
axes[2].tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('Store Budget Allocation', fontsize=12)
plt.tight_layout(); plt.show()


---
## Module 3 — Markdown Scheduling

In [ ]:
depths = MARKDOWN_DEPTHS
lifts  = [demand_lift(d, elasticity=-2.5) for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(depths*100, lifts, color=COLORS['green'], marker='o', lw=2)
axes[0].axhline(1, color=COLORS['gray'], lw=1, linestyle='--')
axes[0].set_title('Demand Lift vs Markdown Depth  (epsilon=-2.5)')
axes[0].set_xlabel('Depth (%)'); axes[0].set_ylabel('Lift factor')
for d, l in zip(depths*100, lifts):
    axes[0].annotate(f'+{(l-1)*100:.0f}%', (d,l),
                     textcoords='offset points', xytext=(0,7), ha='center', fontsize=7)

net = [(1-d)*l for d,l in zip(depths, lifts)]
axes[1].plot(depths*100, net, color=COLORS['orange'], marker='o', lw=2)
axes[1].axhline(1, color=COLORS['gray'], lw=1, linestyle='--', label='Baseline')
axes[1].set_title('Net Revenue Factor vs Markdown Depth')
axes[1].set_xlabel('Depth (%)'); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
md = markdown_dollar_impact(fc)
md_detail = md.pop('item_detail_df')
print('=== Markdown Dollar Impact ===')
for k, v in md.items():
    print(f'  {k:<45}: {fmt(v) if isinstance(v,(int,float)) else v}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
marked = md_detail[md_detail['markdown_applied']]

depth_counts = marked['markdown_depth_pct'].value_counts().sort_index()
axes[0].bar(depth_counts.index.astype(str), depth_counts.values,
            color=COLORS['orange'], edgecolor='none')
axes[0].set_title('Markdown Depth Distribution'); axes[0].set_xlabel('Depth (%)')

s = marked.sample(min(1000, len(marked)), random_state=42)
sc = axes[1].scatter(s['excess_units'], s['rev_gain_usd'],
                     c=s['markdown_depth_pct'], cmap='RdYlGn', s=15, alpha=0.6)
plt.colorbar(sc, ax=axes[1], label='Depth %')
axes[1].set_title('Revenue Gain vs Excess Units')
axes[1].set_xlabel('Excess Units'); axes[1].set_ylabel('Revenue Gain ($)')

axes[2].hist(marked['clearance_rate'], bins=40, color=COLORS['purple'], edgecolor='none')
axes[2].axvline(0.8, color=COLORS['orange'], lw=1.5, linestyle='--', label='80% target')
axes[2].set_title('Clearance Rate Distribution'); axes[2].legend()

plt.suptitle('Markdown Scheduling Analysis', fontsize=12)
plt.tight_layout(); plt.show()


---
## Combined Business Impact

In [ ]:
combined = (nv['enterprise_saving_usd']
            + ba['enterprise_uplift_usd']
            + md['enterprise_gain_usd'])

print('=' * 56)
print('ENTERPRISE ANNUAL IMPACT  (4,700 US stores)')
print('=' * 56)
print(f'  Module 1 — Newsvendor (inventory cost) : {fmt(nv["enterprise_saving_usd"])}')
print(f'  Module 2 — Budget allocation (revenue) : {fmt(ba["enterprise_uplift_usd"])}')
print(f'  Module 3 — Markdown scheduling (revenue): {fmt(md["enterprise_gain_usd"])}')
print('-' * 56)
print(f'  TOTAL                                  : {fmt(combined)}')
print('=' * 56)


In [ ]:
modules   = ['Newsvendor\n(Inventory)','Budget\nAllocation','Markdown\nScheduling']
ann_vals  = [nv['enterprise_saving_usd'], ba['enterprise_uplift_usd'], md['enterprise_gain_usd']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bars = axes[0].bar(modules, ann_vals,
                   color=[COLORS['green'],COLORS['blue'],COLORS['orange']], edgecolor='none')
for bar, v in zip(bars, ann_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 fmt(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Enterprise Annual Impact by Module\n(4,700 Walmart US stores)', fontsize=11)
axes[0].set_ylabel('Annual Impact ($)')

# Waterfall
labels  = ['Newsvendor','Budget\nAlloc','Markdown','Total']
running = 0
bottoms = []
for v in ann_vals:
    bottoms.append(running); running += v
bottoms.append(0)
wf_colors = [COLORS['green'],COLORS['blue'],COLORS['orange'],COLORS['purple']]
for i,(label,val,bot,col) in enumerate(zip(labels, ann_vals+[combined], bottoms, wf_colors)):
    axes[1].bar(label, val, bottom=bot, color=col, edgecolor='none', alpha=0.9)
    axes[1].text(i, bot+val/2, fmt(val), ha='center', va='center',
                 fontsize=9, fontweight='bold', color='white')
running = 0
for i, v in enumerate(ann_vals[:-1]):
    running += v
    axes[1].plot([i+0.4, i+0.6],[running,running],
                 color=COLORS['gray'], lw=1, linestyle='--')
axes[1].set_title(f'Cumulative Waterfall — Total: {fmt(combined)}', fontsize=11)
axes[1].set_ylabel('Annual Impact ($)')

plt.suptitle('Combined Enterprise Business Impact', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
summary = pd.DataFrame([
    {'Module':'Newsvendor',        '28d':fmt(nv['saving_28d_usd']),           'Enterprise Annual':fmt(nv['enterprise_saving_usd'])},
    {'Module':'Budget Allocation', '28d':fmt(ba['revenue_uplift_28d_usd']),   'Enterprise Annual':fmt(ba['enterprise_uplift_usd'])},
    {'Module':'Markdown',          '28d':fmt(md['revenue_gain_28d_usd']),     'Enterprise Annual':fmt(md['enterprise_gain_usd'])},
    {'Module':'COMBINED',          '28d':'—',                                  'Enterprise Annual':fmt(combined)},
]).set_index('Module')
print(summary.to_string())
print()
print('Pipeline complete!')
print('01_eda -> 02_feature_engineering -> 03_forecasting -> 04_optimization')
